# Download optional full-text PDFs — run this where you HAVE access

Colab's IP is blocked by paywalled publishers (MDPI, OUP, Wiley, ASCO, IEEE, Elsevier…),
so the **main notebook** screens those papers on the **sheet abstract** only. Run **this**
notebook on a machine with institutional/library access — your **university network** — to
grab just those PDFs, then zip them and upload the zip back to the main notebook.

**The loop**

1. Main notebook, Step 2b → writes `data/missing_fulltext.csv` (the blocked papers) and
   downloads it to you.
2. Open **this** notebook on the university network → upload that CSV → it downloads each
   paper's PDF into `manual_pdfs/`.
3. It zips them to **`manual_pdfs.zip`** → upload that in the main notebook's re-import
   cell → re-run Step 2 → the blocked papers now screen on **full text**.

No GPU, no API keys, no Ollama — just `requests`. Works in Colab or plain Jupyter.

In [ ]:
# --- self-contained setup: clone the repo + install the few deps we need ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/putssander/systematic-review-pipeline-pharma.git"
REPO_DIR = "systematic-review-pipeline-pharma"
if not Path("pipeline").exists():
    if not Path(REPO_DIR).exists():
        subprocess.run(f"git clone -q {REPO}", shell=True)
    if Path(REPO_DIR, "pipeline").exists():
        os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "beautifulsoup4", "tqdm", "pandas"])
print("Ready in", Path.cwd(), "| Colab:", IN_COLAB)

## 1) Provide the worklist of blocked papers

Use `data/missing_fulltext.csv` that the main notebook wrote (Step 2b). On Colab, run the
cell and pick that file. In plain Jupyter, just copy it into this repo's `data/` folder.

*(No worklist? You can still run step 2 with `do_all=True` to fetch every paper.)*

In [ ]:
from pipeline import config
if IN_COLAB and not config.MISSING_CSV.exists():
    from google.colab import files
    print("Upload the missing_fulltext.csv from the main notebook:")
    for name, data in files.upload().items():
        if name.lower().endswith(".csv"):
            config.MISSING_CSV.write_bytes(data)

if config.MISSING_CSV.exists():
    import pandas as pd
    wl = pd.read_csv(config.MISSING_CSV)
    print(f"worklist: {len(wl)} paper(s) to fetch")
    display(wl)
else:
    print("No worklist found — run step 2 below with do_all=True to fetch all papers.")

## 2) Download the PDFs

For each paper it fetches the resolved URL and, if that's a landing page, follows the
`citation_pdf_url` the publisher advertises. Anything it can't get (login-walled, no PDF
link) is listed in `manual_pdfs/_still_missing.csv` with the link so you can save those by
hand as `<record_id>.pdf`.

In [ ]:
from pipeline import download_pdfs
# Uses data/missing_fulltext.csv by default. Set do_all=True to fetch every paper.
ok, n_missing, still = download_pdfs.run(do_all=False)

## 3) Zip it and take it back to the main notebook

Upload the `manual_pdfs.zip` this produces into the main notebook's **Step 2b re-import**
cell, then re-run Step 2 there.

In [ ]:
import shutil
from pipeline import config
zip_path = shutil.make_archive("manual_pdfs", "zip", config.PDF_DROP_DIR)
print("wrote", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)